<a href="https://colab.research.google.com/github/SukritiChauhan/cifar10-uncertainty-benchmark/blob/main/02_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(mean, std)])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
test_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

print(len(train_set), len(test_set))
x, y = train_set[0]
print(x.shape, y)

100%|██████████| 170M/170M [45:02<00:00, 63.1kB/s]


50000 10000
torch.Size([3, 32, 32]) 6


In [5]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r ./data /content/drive/MyDrive/cifar_data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 64), block(64, 128), block(128, 256))
        self.drop = nn.Dropout(p)
        self.fc = nn.Linear(256 * 4 * 4, 10)

    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.fc(self.drop(x))

In [8]:
model = SmallCNN().to(device)
x, y = next(iter(train_loader))
out = model(x.to(device))
print(out.shape)
print(sum(p.numel() for p in model.parameters()))

torch.Size([128, 10])
1187274


In [9]:
def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

def train(model, epochs=15, lr=0.05):
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader))
    for ep in range(epochs):
        model.train(); running = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward(); opt.step(); sched.step()
            running += loss.item()
        print(f"epoch {ep+1}: loss {running/len(train_loader):.3f}, "
              f"test acc {evaluate(model, test_loader):.3f}")

print("untrained accuracy:", evaluate(SmallCNN().to(device), test_loader))
model = SmallCNN()
train(model)
torch.save(model.state_dict(), "baseline.pt")

untrained accuracy: 0.1
epoch 1: loss 1.433, test acc 0.587
epoch 2: loss 0.994, test acc 0.709
epoch 3: loss 0.794, test acc 0.681
epoch 4: loss 0.670, test acc 0.744
epoch 5: loss 0.589, test acc 0.783
epoch 6: loss 0.528, test acc 0.811
epoch 7: loss 0.480, test acc 0.810
epoch 8: loss 0.440, test acc 0.806
epoch 9: loss 0.401, test acc 0.846
epoch 10: loss 0.360, test acc 0.849
epoch 11: loss 0.326, test acc 0.866
epoch 12: loss 0.286, test acc 0.885
epoch 13: loss 0.235, test acc 0.896
epoch 14: loss 0.196, test acc 0.901
epoch 15: loss 0.167, test acc 0.903


In [10]:
!cp baseline.pt /content/drive/MyDrive/


In [11]:
def get_probs(model, loader):
    model.eval(); probs, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            p = F.softmax(model(x.to(device)), dim=1)
            probs.append(p.cpu()); labels.append(y)
    return torch.cat(probs), torch.cat(labels)

def ece(probs, labels, n_bins=15):
    conf, pred = probs.max(1)
    correct = (pred == labels).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            gap = (conf[mask].mean() - correct[mask].mean()).abs()
            total += mask.float().mean() * gap
    return total.item()

probs, labels = get_probs(model, test_loader)
acc = (probs.argmax(1) == labels).float().mean().item()
nll = F.nll_loss(probs.clamp_min(1e-12).log(), labels).item()
print(f"accuracy {acc:.3f} | avg confidence {probs.max(1).values.mean():.3f} | "
      f"ECE {ece(probs, labels):.3f} | NLL {nll:.3f}")

accuracy 0.903 | avg confidence 0.930 | ECE 0.028 | NLL 0.294


In [12]:
import random, numpy as np

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

members = []
for seed in [0, 1, 2]:
    set_seed(seed)
    m = SmallCNN()
    print(f"--- training member {seed} ---")
    train(m)
    torch.save(m.state_dict(), f"member_{seed}.pt")
    members.append(m)

--- training member 0 ---
epoch 1: loss 1.405, test acc 0.607
epoch 2: loss 0.982, test acc 0.632
epoch 3: loss 0.792, test acc 0.728
epoch 4: loss 0.674, test acc 0.774
epoch 5: loss 0.589, test acc 0.778
epoch 6: loss 0.530, test acc 0.811
epoch 7: loss 0.481, test acc 0.804
epoch 8: loss 0.435, test acc 0.814
epoch 9: loss 0.403, test acc 0.835
epoch 10: loss 0.362, test acc 0.860
epoch 11: loss 0.325, test acc 0.862
epoch 12: loss 0.282, test acc 0.880
epoch 13: loss 0.234, test acc 0.892
epoch 14: loss 0.192, test acc 0.901
epoch 15: loss 0.164, test acc 0.903
--- training member 1 ---
epoch 1: loss 1.440, test acc 0.611
epoch 2: loss 0.978, test acc 0.662
epoch 3: loss 0.788, test acc 0.689
epoch 4: loss 0.668, test acc 0.730
epoch 5: loss 0.589, test acc 0.747
epoch 6: loss 0.531, test acc 0.813
epoch 7: loss 0.484, test acc 0.786
epoch 8: loss 0.441, test acc 0.806
epoch 9: loss 0.405, test acc 0.834
epoch 10: loss 0.369, test acc 0.833
epoch 11: loss 0.327, test acc 0.865
epoc

In [13]:
def report(name, probs, labels):
    acc = (probs.argmax(1) == labels).float().mean().item()
    nll = F.nll_loss(probs.clamp_min(1e-12).log(), labels).item()
    print(f"{name:15s} acc {acc:.3f} | conf {probs.max(1).values.mean():.3f} | "
          f"ECE {ece(probs, labels):.3f} | NLL {nll:.3f}")

results = [get_probs(m, test_loader) for m in members]
labels = results[0][1]
all_probs = [r[0] for r in results]

for i, p in enumerate(all_probs):
    report(f"single model {i}", p, labels)
report("ensemble of 3", torch.stack(all_probs).mean(0), labels)

single model 0  acc 0.903 | conf 0.932 | ECE 0.030 | NLL 0.297
single model 1  acc 0.905 | conf 0.929 | ECE 0.025 | NLL 0.292
single model 2  acc 0.903 | conf 0.927 | ECE 0.025 | NLL 0.298
ensemble of 3   acc 0.915 | conf 0.917 | ECE 0.010 | NLL 0.253


In [14]:
!cp member_*.pt /content/drive/MyDrive/

In [15]:
def get_probs_mc(model, loader, T=20):
    model.eval()
    model.drop.train()  # dropout stays on; BatchNorm stays in eval mode
    probs, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            feats = model.features(x.to(device)).flatten(1)
            p = torch.stack([F.softmax(model.fc(model.drop(feats)), dim=1)
                             for _ in range(T)]).mean(0)
            probs.append(p.cpu()); labels.append(y)
    return torch.cat(probs), torch.cat(labels)

mc_probs, labels = get_probs_mc(members[0], test_loader)
report("MC dropout T=20", mc_probs, labels)

MC dropout T=20 acc 0.902 | conf 0.924 | ECE 0.023 | NLL 0.291


In [16]:
def get_logits(model, loader):
    model.eval(); outs, labs = [], []
    with torch.no_grad():
        for x, y in loader:
            outs.append(model(x.to(device)).cpu()); labs.append(y)
    return torch.cat(outs), torch.cat(labs)

def best_T(logits, labels):
    Ts = torch.linspace(0.5, 3.0, 251)
    nlls = [F.cross_entropy(logits / t, labels).item() for t in Ts]
    return Ts[int(torch.tensor(nlls).argmin())].item()

logits, labs = get_logits(members[0], test_loader)
fit_idx = torch.arange(0, 10000, 2)    # half A: used to find T
eval_idx = torch.arange(1, 10000, 2)   # half B: used to report

T = best_T(logits[fit_idx], labs[fit_idx])
print("best temperature:", round(T, 2))
report("before (half B)", F.softmax(logits[eval_idx], 1), labs[eval_idx])
report("after  (half B)", F.softmax(logits[eval_idx] / T, 1), labs[eval_idx])

best temperature: 1.34
before (half B) acc 0.904 | conf 0.932 | ECE 0.028 | NLL 0.293
after  (half B) acc 0.904 | conf 0.903 | ECE 0.007 | NLL 0.278


In [17]:
for i, m in enumerate(members):
    lg, lb = get_logits(m, test_loader)
    Ti = best_T(lg[fit_idx], lb[fit_idx])
    report(f"seed {i} before", F.softmax(lg[eval_idx], 1), lb[eval_idx])
    report(f"seed {i} T={Ti:.2f}", F.softmax(lg[eval_idx] / Ti, 1), lb[eval_idx])

seed 0 before   acc 0.904 | conf 0.932 | ECE 0.028 | NLL 0.293
seed 0 T=1.34   acc 0.904 | conf 0.903 | ECE 0.007 | NLL 0.278
seed 1 before   acc 0.909 | conf 0.931 | ECE 0.025 | NLL 0.286
seed 1 T=1.30   acc 0.909 | conf 0.906 | ECE 0.010 | NLL 0.274
seed 2 before   acc 0.906 | conf 0.929 | ECE 0.026 | NLL 0.291
seed 2 T=1.30   acc 0.906 | conf 0.903 | ECE 0.008 | NLL 0.279


In [18]:
def line(name, probs):
    report(name, probs[eval_idx], labs[eval_idx])

ens_probs = torch.stack(all_probs).mean(0)

for i, m in enumerate(members):
    lg, _ = get_logits(m, test_loader)
    Ti = best_T(lg[fit_idx], labs[fit_idx])
    line(f"seed {i} single", F.softmax(lg, 1))
    line(f"seed {i} + temp T={Ti:.2f}", F.softmax(lg / Ti, 1))
    mc, _ = get_probs_mc(m, test_loader)
    line(f"seed {i} MC dropout", mc)

line("ensemble of 3", ens_probs)
lg_e = ens_probs.clamp_min(1e-12).log()
Te = best_T(lg_e[fit_idx], labs[fit_idx])
line(f"ensemble + temp T={Te:.2f}", F.softmax(lg_e / Te, 1))

seed 0 single   acc 0.904 | conf 0.932 | ECE 0.028 | NLL 0.293
seed 0 + temp T=1.34 acc 0.904 | conf 0.903 | ECE 0.007 | NLL 0.278
seed 0 MC dropout acc 0.905 | conf 0.925 | ECE 0.021 | NLL 0.288
seed 1 single   acc 0.909 | conf 0.931 | ECE 0.025 | NLL 0.286
seed 1 + temp T=1.30 acc 0.909 | conf 0.906 | ECE 0.010 | NLL 0.274
seed 1 MC dropout acc 0.910 | conf 0.924 | ECE 0.019 | NLL 0.283
seed 2 single   acc 0.906 | conf 0.929 | ECE 0.026 | NLL 0.291
seed 2 + temp T=1.30 acc 0.906 | conf 0.903 | ECE 0.008 | NLL 0.279
seed 2 MC dropout acc 0.905 | conf 0.921 | ECE 0.017 | NLL 0.286
ensemble of 3   acc 0.916 | conf 0.918 | ECE 0.011 | NLL 0.247
ensemble + temp T=1.08 acc 0.916 | conf 0.910 | ECE 0.010 | NLL 0.246


In [19]:
!cp member_*.pt /content/drive/MyDrive/